In [ ]:
import json
import random
import pickle
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ---------------- CONFIG ---------------- #

INTENTS = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["hi", "hello", "hey", "good morning", "good day"],
            "responses": ["Hello!", "Hi there!", "Hey! How can I help?"]
        },
        {
            "tag": "goodbye",
            "patterns": ["bye", "see you later", "goodbye", "take care"],
            "responses": ["Goodbye!", "See you soon!", "Take care!"]
        },
        {
            "tag": "name",
            "patterns": ["what is your name", "who are you", "tell me your name"],
            "responses": ["I am Zak-Bot, your AI assistant.", "You can call me Zak-Bot!"]
        },
        {
            "tag": "status",
            "patterns": ["how are you", "how do you feel", "are you okay"],
            "responses": ["I'm doing well, thanks!", "All good here. How about you?"]
        },
        {
            "tag": "acknowledgement",
            "patterns": ["okay", "ok", "got it", "fine", "sure"],
            "responses": ["Great!", "Okay!", "Understood."]
        },
        {
            "tag": "help",
            "patterns": ["can you help me", "i need help", "what can you do"],
            "responses": ["Sure, I can help with basic queries.", "I can chat and answer simple questions."]
        }
    ]
}

MODEL_PATH = "chatbot_model.h5"
WORDS_PATH = "words.pkl"
CLASSES_PATH = "classes.pkl"
THRESHOLD = 0.60

lemmatizer = WordNetLemmatizer()

# ---------------- NLTK SETUP ---------------- #

def ensure_nltk():
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    nltk.download("wordnet", quiet=True)

# ---------------- PREPROCESSING ---------------- #

def tokenize(sentence):
    tokens = nltk.word_tokenize(sentence.lower())
    return [lemmatizer.lemmatize(w) for w in tokens if w.isalpha()]

def bag_of_words(sentence, words):
    sentence_words = tokenize(sentence)
    return np.array([1 if w in sentence_words else 0 for w in words], dtype=np.float32)

# ---------------- DATA PREP ---------------- #

def build_training_data(intents_data):
    words = []
    classes = []
    documents = []

    for intent in intents_data["intents"]:
        tag = intent["tag"]
        classes.append(tag)

        for pattern in intent["patterns"]:
            word_list = tokenize(pattern)
            words.extend(word_list)
            documents.append((word_list, tag))

    words = sorted(set(words))
    classes = sorted(set(classes))

    X = []
    y = []
    output_empty = [0] * len(classes)

    for word_list, tag in documents:
        bow = np.array([1 if w in word_list else 0 for w in words], dtype=np.float32)
        X.append(bow)

        output_row = output_empty.copy()
        output_row[classes.index(tag)] = 1
        y.append(output_row)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    idx = np.arange(len(X))
    np.random.shuffle(idx)
    X, y = X[idx], y[idx]

    return X, y, words, classes

# ---------------- MODEL ---------------- #

def create_model(input_dim, output_dim):
    model = Sequential([
        Dense(128, input_shape=(input_dim,), activation="relu"),
        Dropout(0.4),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(output_dim, activation="softmax")
    ])
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

def train_and_save(intents_data):
    X, y, words, classes = build_training_data(intents_data)

    model = create_model(X.shape[1], y.shape[1])
    es = EarlyStopping(monitor="loss", patience=20, restore_best_weights=True)

    model.fit(X, y, epochs=300, batch_size=8, verbose=1, callbacks=[es])

    model.save(MODEL_PATH)
    pickle.dump(words, open(WORDS_PATH, "wb"))
    pickle.dump(classes, open(CLASSES_PATH, "wb"))

    return model, words, classes

# ---------------- PREDICTION ---------------- #

def load_bot():
    model = load_model(MODEL_PATH)
    words = pickle.load(open(WORDS_PATH, "rb"))
    classes = pickle.load(open(CLASSES_PATH, "rb"))
    return model, words, classes

def predict_class(sentence, model, words, classes):
    bow = bag_of_words(sentence, words)
    probs = model.predict(np.array([bow]), verbose=0)[0]

    best_idx = int(np.argmax(probs))
    best_prob = float(probs[best_idx])

    if best_prob < THRESHOLD:
        return None, best_prob

    return classes[best_idx], best_prob

def get_response(sentence, model, words, classes, intents_data):
    tag, confidence = predict_class(sentence, model, words, classes)

    if tag is None:
        return "Sorry, I didn't understand that."

    for intent in intents_data["intents"]:
        if intent["tag"] == tag:
            return random.choice(intent["responses"])

    return "Sorry, I didn't understand that."

# ---------------- MAIN ---------------- #

if __name__ == "__main__":
    ensure_nltk()

    try:
        model, words, classes = load_bot()
        print("Loaded saved model.")
    except:
        print("Training new model...")
        model, words, classes = train_and_save(INTENTS)

    print("Zak-Bot ready. Type 'exit' to stop.\n")

    while True:
        text = input("You: ").strip()
        if text.lower() == "exit":
            break
        reply = get_response(text, model, words, classes, INTENTS)
        print(f"Zak-Bot: {reply}")


Training new model...
Epoch 1/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2609 - loss: 1.7468
Epoch 2/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1304 - loss: 1.8124    
Epoch 3/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1739 - loss: 1.7398
Epoch 4/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2174 - loss: 1.7827    
Epoch 5/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3478 - loss: 1.6905
Epoch 6/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4348 - loss: 1.6724 
Epoch 7/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3913 - loss: 1.6769
Epoch 8/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6522 - loss: 1.6415
Epoch 9/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5217 - loss: 1.6267
Epoch 10/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5652 - loss: 1.5987
Epoch 11/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7391 - loss: 1.5523
Epoch 12/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11

Zak-Bot ready. Type 'exit' to stop.

Zak-Bot: Hello!
Zak-Bot: I'm doing well, thanks!
Zak-Bot: I can chat and answer simple questions.
